# **Purpose**

This Notebook Tests and Evaluates the Fine Tuned TinyLlama 1.1B Model for MCQ Question Answer Generation

## **Install the required modules**

In [7]:
!pip install -qU \
evaluate\
nltk\
transformers\
torch\
torchvision\
torchaudio\
datasets\
spacy\
rouge_score

  Preparing metadata (setup.py) ... done


In [8]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Restart the session after all the modules are installed.**

##**Import the required Libraries**

In [16]:
import os
import re
import math
import spacy
import torch
import nltk
from collections import Counter
from google.colab import drive
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, GPT2LMHeadModel, GPT2Tokenizer, GPT2TokenizerFast
import evaluate

In [9]:
!export NLTK_ALLOW_PROXIED_URLOPEN=1 && python -m nltk.downloader punkt_tab punkt wordnet

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


##**Automated NLP Metrics Helper Functions**

In [10]:
def compute_token_f1(pred_str, ref_str):
    pred_tokens = pred_str.lower().split()
    ref_tokens = ref_str.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(ref_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def compute_perplexity(texts, model_id="gpt2"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ppl_tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    ppl_model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    ppl_model.eval()

    nlls = []
    for text in texts:
        if not text.strip():
            continue
        encodings = ppl_tokenizer(text, return_tensors="pt")
        input_ids = encodings.input_ids.to(device)

        with torch.no_grad():
            outputs = ppl_model(input_ids, labels=input_ids)
            nlls.append(outputs.loss)

    if not nlls:
        return 0.0

    return torch.exp(torch.stack(nlls).mean()).item()


def compute_distinct_n(texts, n=1):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        tokens = text.lower().split()
        if len(tokens) < n:
            continue
        ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    return len(unique_ngrams) / total_ngrams if total_ngrams > 0 else 0.0

## **Item-Writing Flaw (IWF) Detection Module**

In [11]:
class IWFDetector:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.vague_terms = {
            "usually", "often", "frequently", "sometimes",
            "generally", "rarely", "seldom", "occasionally"
        }
        self.absolute_terms = {
            "always", "never", "all", "none", "must", "completely", "entirely"
        }

    def check_stem_clues(self, stem, options):
        flaws = []
        stem_lower = stem.lower()
        for opt in options:
            if opt.lower() in stem_lower and len(opt.strip()) > 3:
                flaws.append(f"Answer Leaked in Stem: Option '{opt}' appears verbatim in question.")
        return flaws

    def check_all_or_none_above(self, options):
        flaws = []
        patterns = [
            r"\ball of the above\b",
            r"\bnone of the above\b",
            r"\bboth [a-d] and [a-d]\b",
            r"\ball of these\b"
        ]
        for opt in options:
            for pat in patterns:
                if re.search(pat, opt, re.IGNORECASE):
                    flaws.append(f"Inclusive/Exclusive Option Flaw: Detected '{opt}'.")
        return flaws

    def check_vague_or_absolute_terms(self, stem, options):
        flaws = []
        all_text = " ".join([stem] + options).lower().split()
        for word in all_text:
            cleaned = re.sub(r'[^\w\s]', '', word)
            if cleaned in self.vague_terms:
                flaws.append(f"Vague Frequency Term: Detected '{cleaned}'.")
            elif cleaned in self.absolute_terms:
                flaws.append(f"Absolute Word Cue: Detected '{cleaned}'.")
        return flaws

    def check_grammatical_agreement(self, stem, options):
        flaws = []
        stem_stripped = stem.strip().rstrip("?:_ ")
        last_word = stem_stripped.split()[-1].lower() if stem_stripped.split() else ""

        if last_word in ["a", "an"]:
            for opt in options:
                first_word = opt.strip().split()[0].lower() if opt.strip().split() else ""
                if first_word:
                    first_char = first_word[0]
                    if last_word == "a" and first_char in "aeiou":
                        flaws.append(f"Grammar Cue: Stem ends with 'a' but option '{opt}' starts with a vowel.")
                    elif last_word == "an" and first_char not in "aeiou":
                        flaws.append(f"Grammar Cue: Stem ends with 'an' but option '{opt}' starts with a consonant.")
        return flaws

    def check_length_outliers(self, correct_answer, distractors):
        flaws = []
        all_options = [correct_answer] + distractors
        lengths = [len(opt.split()) for opt in all_options]
        avg_len = sum(lengths) / len(lengths)
        ans_len = len(correct_answer.split())

        if ans_len > 2 * avg_len and ans_len > 4:
            flaws.append("Length Cue: Correct answer is significantly longer than distractors.")
        return flaws

    def evaluate_mcq(self, question_stem, correct_answer, distractors=None):
        if distractors is None:
            distractors = []

        all_options = [correct_answer] + distractors
        detected_flaws = []

        detected_flaws.extend(self.check_all_or_none_above(all_options))
        detected_flaws.extend(self.check_stem_clues(question_stem, [correct_answer]))
        detected_flaws.extend(self.check_vague_or_absolute_terms(question_stem, all_options))
        detected_flaws.extend(self.check_grammatical_agreement(question_stem, all_options))
        if distractors:
            detected_flaws.extend(self.check_length_outliers(correct_answer, distractors))

        return {
            "is_valid": len(detected_flaws) == 0,
            "flaw_count": len(detected_flaws),
            "flaws": list(set(detected_flaws))
        }

##**Main Evaluation Pipeline**

In [12]:
def run_full_evaluation(merged_model_path, num_samples=1500):
    # Mount Google Drive
    drive.mount('/content/drive', force_remount=True)

    print(f"Loading model and tokenizer from: {merged_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
    model = AutoModelForCausalLM.from_pretrained(
        merged_model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    qg_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
        max_new_tokens=128,
        do_sample=False
    )

    print(f"Loading SQuAD v2 validation dataset ({num_samples} samples)...")
    val_dataset = load_dataset('rajpurkar/squad_v2', split='validation').shuffle(seed=42).select(range(num_samples))

    references = []
    predictions = []
    target_answers = []

    print("Generating question predictions...")
    for example in val_dataset:
        context = example["context"]
        answers = example.get("answers", {})

        if not answers or len(answers.get("text", [])) == 0:
            continue

        target_answer = answers["text"][0]
        true_question = example["question"]

        messages = [
            {
                "role": "system",
                "content": "You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer."
            },
            {
                "role": "user",
                "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
            }
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = qg_pipeline(prompt)
        generated_text = output[0]['generated_text']

        # Extract generated question text
        pred_question = generated_text.split("Answer:")[0].replace("Question:", "").strip()

        predictions.append(pred_question)
        references.append(true_question)
        target_answers.append(target_answer)

    # -------------------------------------------------------------
    # Compute NLP Overlap Metrics
    # -------------------------------------------------------------
    print("Computing NLP benchmark metrics...")
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")

    bleu_res = bleu_metric.compute(predictions=predictions, references=references)
    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    meteor_res = meteor_metric.compute(predictions=predictions, references=references)
    avg_f1 = sum(compute_token_f1(p, r) for p, r in zip(predictions, references)) / len(predictions)

    # -------------------------------------------------------------
    # Compute Language Quality & Diversity Metrics
    # -------------------------------------------------------------
    ppl = compute_perplexity(predictions)
    dist1 = compute_distinct_n(predictions, n=1)
    dist2 = compute_distinct_n(predictions, n=2)

    # -------------------------------------------------------------
    # Compute IWF Metrics on Predictions
    # -------------------------------------------------------------
    print("Running Item-Writing Flaws (IWF) detection...")
    detector = IWFDetector()
    valid_count = 0
    total_flaws = 0

    for q, a in zip(predictions, target_answers):
        res = detector.evaluate_mcq(question_stem=q, correct_answer=a)
        if res["is_valid"]:
            valid_count += 1
        total_flaws += res["flaw_count"]

    iwf_pass_rate = (valid_count / len(predictions)) * 100

    # -------------------------------------------------------------
    # Display Benchmark Summary
    # -------------------------------------------------------------
    print("\n" + "=" * 50)
    print("      M.TECH PROJECT EVALUATION RESULTS")
    print("=" * 50)
    print(f"Evaluated Samples        : {len(predictions)}")
    print("-" * 50)
    print(f"BLEU Score               : {bleu_res['bleu']:.4f}")
    print(f"ROUGE-1                  : {rouge_res['rouge1']:.4f}")
    print(f"ROUGE-2                  : {rouge_res['rouge2']:.4f}")
    print(f"ROUGE-L                  : {rouge_res['rougeL']:.4f}")
    print(f"METEOR Score             : {meteor_res['meteor']:.4f}")
    print(f"Average Token F1         : {avg_f1:.4f}")
    print("-" * 50)
    print(f"Perplexity (GPT-2)       : {ppl:.2f}")
    print(f"Distinct-1 (Diversity)   : {dist1:.4f}")
    print(f"Distinct-2 (Diversity)   : {dist2:.4f}")
    print("-" * 50)
    print(f"IWF Pass Rate            : {iwf_pass_rate:.2f}%")
    print(f"Total Structural Flaws   : {total_flaws}")
    print("=" * 50)

##**Execution Entry Point**


In [17]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

import nltk
# Pre-download all required corpora and tokenizers
for resource in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords']:
    nltk.download(resource, quiet=True)

if __name__ == "__main__":
    DRIVE_MODEL_PATH = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq"
    run_full_evaluation(merged_model_path=DRIVE_MODEL_PATH, num_samples=200)

Mounted at /content/drive
Loading model and tokenizer from: /content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading SQuAD v2 validation dataset (200 samples)...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating question predictions...


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Running Item-Writing Flaws (IWF) detection...

      M.TECH PROJECT EVALUATION RESULTS
Evaluated Samples        : 97
--------------------------------------------------
BLEU Score               : 0.1643
ROUGE-1                  : 0.4279
ROUGE-2                  : 0.2068
ROUGE-L                  : 0.3934
METEOR Score             : 0.3958
Average Token F1         : 0.3864
--------------------------------------------------
Perplexity (GPT-2)       : 70.04
Distinct-1 (Diversity)   : 0.4599
Distinct-2 (Diversity)   : 0.7466
--------------------------------------------------
IWF Pass Rate            : 89.69%
Total Structural Flaws   : 10
